In [3]:
memory = {}

def write_slot(entity, attribute, value):
    if entity not in memory:
        memory[entity] = {}

    # TODO 1: Store value under its attribute
    memory[entity][attribute] = value

def read_slot(entity, attribute):
    # TODO 2: Retrieve the selected attribute
    return memory[entity][attribute]


write_slot("dragon", "color", "red")
write_slot("dragon", "age", 500)
write_slot("dragon", "element", "fire")

print(read_slot("dragon", "color"))
print(read_slot("dragon", "age"))
print(read_slot("dragon", "element"))
write_slot("dragon", "age", 600)

print(memory["dragon"])

red
500
fire
{'color': 'red', 'age': 600, 'element': 'fire'}


In [4]:
import torch

vector_memory = {}

def write_vector(entity, attribute, value):
    if entity not in vector_memory:
        vector_memory[entity] = {}

    # TODO 1:
    # Store a copy of the value tensor
    vector_memory[entity][attribute] = torch.clone(value)


def read_vector(entity, attribute):
    # TODO 2:
    # Return a copy of the selected tensor
    return torch.clone(vector_memory[entity][attribute])


# Three independently stored values
v_color = torch.tensor([1.0, 0.0, 0.0])
v_age = torch.tensor([0.5, 0.2, 0.8])
v_element = torch.tensor([0.0, 1.0, 1.0])

write_vector("dragon", "color", v_color)
write_vector("dragon", "age", v_age)
write_vector("dragon", "element", v_element)

# Retrieve
color = read_vector("dragon", "color")
age = read_vector("dragon", "age")
element = read_vector("dragon", "element")

print("Color:", color)
print("Age:", age)
print("Element:", element)

# Retrieval fidelity
print("Color exact:", torch.allclose(color, v_color))
print("Age exact:", torch.allclose(age, v_age))
print("Element exact:", torch.allclose(element, v_element))

Color: tensor([1., 0., 0.])
Age: tensor([0.5000, 0.2000, 0.8000])
Element: tensor([0., 1., 1.])
Color exact: True
Age exact: True
Element exact: True


In [5]:
import torch

# Our three target values
values = {
    "color": v_color,
    "age": v_age,
    "element": v_element
}

# Three key vectors in 2D space
keys = {
    "color": torch.tensor([1., 0.]),
    "age": torch.tensor([0., 1.]),
    "element": torch.tensor([2**-0.5, 2**-0.5])
}

S = torch.zeros(2, 3)
beta = 1.0

# Write all three associations
for name in values:
    k = keys[name]
    v = values[name]

    prediction = k @ S
    error = v - prediction

    S = S + beta * torch.outer(k, error)

# Now check all three associations
for name in values:
    retrieved = keys[name] @ S
    target = values[name]

    retrieval_error = torch.norm(retrieved - target)

    print(f"{name}:")
    print(" Retrieved:", retrieved)
    print(" Target:   ", target)
    print(" L2 error: ", retrieval_error.item())
    print()

color:
 Retrieved: tensor([0.2500, 0.6071, 0.3071])
 Target:    tensor([1., 0., 0.])
 L2 error:  1.0126169919967651

age:
 Retrieved: tensor([-0.2500,  0.8071,  1.1071])
 Target:    tensor([0.5000, 0.2000, 0.8000])
 L2 error:  1.0126169919967651

element:
 Retrieved: tensor([8.6850e-08, 1.0000e+00, 1.0000e+00])
 Target:    tensor([0., 1., 1.])
 L2 error:  1.0533610605989452e-07



In [7]:
import torch

def update_memory(S, k, v, beta=1.0):
    prediction = k @ S
    error = v - prediction
    return S + beta * torch.outer(k, error)

In [8]:
S = torch.zeros(2, 3)

write_order = ["element", "age", "color"]

for name in write_order:
    S = update_memory(
        S,
        keys[name],
        values[name],
        beta=1.0
    )

print("Final memory:\n", S)

for name in values:
    retrieved = keys[name] @ S
    error = torch.norm(retrieved - values[name])

    print(f"{name}: L2 error = {error.item():.6f}")

Final memory:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.2000, 0.8000]])
color: L2 error = 0.000000
age: L2 error = 0.000000
element: L2 error = 1.432057


In [9]:
S = torch.zeros(2, 3)

write_order = ["element", "age", "color"]

for name in write_order:
    S = update_memory(
        S, keys[name], values[name], beta=1.0
    )

for name in values:
    retrieved = keys[name] @ S
    error = torch.norm(retrieved - values[name])

    print(f"{name}: L2 error = {error.item():.6f}")

color: L2 error = 0.000000
age: L2 error = 0.000000
element: L2 error = 1.432057


In [10]:
# Three independent keys
keys3 = {
    "color": torch.tensor([1., 0., 0.]),
    "age": torch.tensor([0., 1., 0.]),
    "element": torch.tensor([0., 0., 1.])
}

# New matrix
S3 = torch.zeros(3, 3)

for name in ["color", "age", "element"]:
    S3 = update_memory(
        S3,
        keys3[name],
        values[name],
        beta=1.0
    )

# Evaluate retrieval
for name in values:
    retrieved = keys3[name] @ S3

    # TODO 1: Calculate L2 retrieval error
    error = torch.norm(retrieved - values[name])

    print(f"{name}: error = {error.item():.8f}")

color: error = 0.00000000
age: error = 0.00000000
element: error = 0.00000000


In [11]:
def tensor_bytes(t):
    return t.numel() * t.element_size()

matrix_2d_bytes = tensor_bytes(S)
matrix_3d_bytes = tensor_bytes(S3)

# TODO 2: Sum the tensor bytes of three values
slots_bytes = sum(
    tensor_bytes(v)
    for v in values.values()
)

print("2D matrix:", matrix_2d_bytes, "bytes")
print("3D matrix:", matrix_3d_bytes, "bytes")
print("Vector slots:", slots_bytes, "bytes")

2D matrix: 24 bytes
3D matrix: 36 bytes
Vector slots: 36 bytes
